# Installing FAISS

FAISS (Facebook AI Similarity Search) is used for
fast semantic similarity search on vector embeddings.

# Research Paper Recommendation System  
## FAISS Similarity Search

This notebook implements efficient semantic similarity search using FAISS.

The workflow includes:

- Loading BERT embeddings
- Building FAISS index
- Performing nearest neighbor search
- Generating semantic recommendations
- Saving FAISS index for deployment

FAISS improves retrieval speed and scalability for large embedding datasets.

In [1]:
!pip install faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
# Load Cleaned Dataset

In [5]:
# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("../Dataset/cleaned_data.csv")
print(df.shape)
df.head()

(5000, 14)


,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count,combined_text,clean_text,char_count,word_count
0,abs-1302.3557v1,Approximations for Decision Making in the Demp...,Artificial Intelligence,cs.AI,2/13/13,2/13/13,['Mathias Bauer'],'Mathias Bauer',The computational complexity of reasoning with...,98,Approximations for Decision Making in the Demp...,approximation decision making dempstershafer t...,544,62
1,abs-1509.03247v1,An Epsilon Hierarchical Fuzzy Twin Support Vec...,Artificial Intelligence,cs.AI,9/10/15,9/10/15,['Arindam Chaudhuri'],'Arindam Chaudhuri',The research presents epsilon hierarchical fuz...,129,An Epsilon Hierarchical Fuzzy Twin Support Vec...,epsilon hierarchical fuzzy twin support vector...,881,104
2,abs-1603.02738v1,Learning to Blend Computer Game Levels,Artificial Intelligence,cs.AI,3/8/16,3/8/16,"['Matthew Guzdial', 'Mark Riedl']",'Matthew Guzdial',We present an approach to generate novel compu...,122,Learning to Blend Computer Game Levels We pres...,learning blend computer game level present app...,651,83
3,abs-1203.3493v1,Solving Hybrid Influence Diagrams with Determi...,Artificial Intelligence,cs.AI,3/15/12,3/15/12,"['Yijing Li', 'Prakash P. Shenoy']",'Yijing Li',We describe a framework and an algorithm for s...,93,Solving Hybrid Influence Diagrams with Determi...,solving hybrid influence diagram deterministic...,604,65
4,abs-1106.0667v1,Reasoning within Fuzzy Description Logics,Artificial Intelligence,cs.AI,6/3/11,6/3/11,['U. Straccia'],'U. Straccia',"Description Logics (DLs) are suitable, well-kn...",133,Reasoning within Fuzzy Description Logics Desc...,reasoning within fuzzy description logic descr...,682,83


In [8]:
# Load Saved BERT Embeddings
# ==========================================
# Load Embeddings
# ==========================================

bert_embeddings = np.load("../Model/bert_embeddings_5k.npy")

print("Embeddings Shape:", bert_embeddings.shape)

Embeddings Shape: (5000, 384)


# Creating FAISS Index

Since embeddings are normalized,
Inner Product becomes Cosine Similarity.

In [9]:
# ==========================================
# Get Embedding Dimension
# ==========================================

embedding_dimension = bert_embeddings.shape[1]

print("Embedding Dimension:", embedding_dimension)

Embedding Dimension: 384


In [10]:
# ==========================================
# Create FAISS Index
# ==========================================

index = faiss.IndexFlatIP(embedding_dimension)

print("FAISS Index Created Successfully")

FAISS Index Created Successfully


# Adding Embeddings to FAISS Index


In [11]:
# ==========================================
# Add Embeddings
# ==========================================

index.add(bert_embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 5000


# Saving FAISS Index

In [12]:
# ==========================================
# Save FAISS Index
# ==========================================

faiss.write_index(index, "faiss_index.index")

print("FAISS Index Saved Successfully!")

FAISS Index Saved Successfully!


# Load BERT Model

In [13]:
# ==========================================
# Load Sentence Transformer
# ==========================================

bert_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("BERT Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BERT Model Loaded


# FAISS Recommendation Function

This function:
1. Converts query into embedding
2. Searches nearest vectors
3. Returns most similar papers

In [14]:
# ==========================================
# FAISS Recommendation Function
# ==========================================

def faiss_recommend(query_text, top_k=5):

    # Convert query into embedding
    query_embedding = bert_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search FAISS index
    similarity_scores, indices = index.search(
        query_embedding,
        top_k
    )

    # Get recommendations
    recommendations = df.iloc[indices[0]][
        ['title', 'summary']
    ].copy()

    # Add similarity scores
    recommendations['similarity_score'] = similarity_scores[0]

    return recommendations

In [15]:
# ==========================================
# TEST FAISS RECOMMENDATION SYSTEM
# ==========================================

query = """
Deep learning methods for image classification
"""

results = faiss_recommend(query)

results

,title,summary,similarity_score
4339,An Algorithm for Quasi-Associative and Quasi-M...,In this paper one proposes a simple algorithm ...,0.446648
2793,GIB: Imperfect Information in a Computationall...,This paper investigates the problems arising i...,0.433669
2109,Causality in Bayesian Belief Networks,We address the problem of causal interpretatio...,0.433226
2749,Indexing the Event Calculus with Kd-trees to M...,Personal Health Systems (PHS) are mobile solut...,0.425007
1297,Direct Uncertainty Estimation in Reinforcement...,Optimal probabilistic approach in reinforcemen...,0.423551


In [16]:
for idx, row in results.iterrows():

    print("=" * 100)

    print("TITLE:\n")
    print(row['title'])

    print("\nSIMILARITY SCORE:")
    print(round(row['similarity_score'], 4))

    print("\nSUMMARY:\n")
    print(row['summary'][:500])

TITLE:

An Algorithm for Quasi-Associative and Quasi-Markovian Rules of
  Combination in Information Fusion

SIMILARITY SCORE:
0.4466

SUMMARY:

In this paper one proposes a simple algorithm of combining the fusion rules,
those rules which first use the conjunctive rule and then the transfer of
conflicting mass to the non-empty sets, in such a way that they gain the
property of associativity and fulfill the Markovian requirement for dynamic
fusion. Also, a new rule, SDL-improved, is presented.
TITLE:

GIB: Imperfect Information in a Computationally Challenging Game

SIMILARITY SCORE:
0.4337

SUMMARY:

This paper investigates the problems arising in the construction of a program
to play the game of contract bridge. These problems include both the difficulty
of solving the game's perfect information variant, and techniques needed to
address the fact that bridge is not, in fact, a perfect information game. GIB,
the program being described, involves five separate technical advances:
partit

In [17]:
results.to_csv("faiss_recommendations.csv", index=False)

print("FAISS Recommendations Saved Successfully!")

FAISS Recommendations Saved Successfully!


# Conclusion

In this notebook:

- BERT embeddings were indexed using FAISS
- Fast similarity search was implemented
- Semantic recommendation retrieval was optimized

FAISS significantly improves recommendation scalability and retrieval efficiency for large NLP embedding datasets.